In [1]:
using Random, Distributions, Statistics, Printf, DelimitedFiles, Dates
using LinearAlgebra
using StatsBase
using QuantileRegressions
using Plots
const bb = 120 
const aa = 40
const N  = 2_701_767
const I0 = 3
const S0 = 2_701_767 - 3 
const n_iter = 1_000_000
include("functions.jl")
Random.seed!(2025)


Istar_obs = [
2, 6, 11, 14, 17, 23, 31, 38, 43, 46, 74, 91, 119, 138, 193, 255, 257,
324, 372, 412, 422, 407, 411, 450, 408, 394, 371, 416, 425, 388, 387,
369, 386, 365, 328, 314, 335, 298, 323, 300, 280, 285, 273, 254, 253,
211, 209, 232, 203, 217, 199, 206, 217, 182, 173, 176, 154, 166, 157
]
tau = length(Istar_obs)

model_tag_sym = :sliding
KMAX_UPPER = 30  
KMAX_fixed = 22

# Fixed output dir
out_dir = "output"
isdir(out_dir) || mkpath(out_dir)

header_cont = []
if model_tag_sym === :memoryless
    global header_cont = ["beta", "alpha", "gamma"]
elseif model_tag_sym === :powerlaw
    global header_cont = ["beta", "alpha", "gamma", "lambda_P"]
elseif model_tag_sym === :exponential
    global header_cont = ["beta", "alpha", "gamma", "lambda_E"]
elseif model_tag_sym === :reciprocal
    global header_cont = ["beta", "alpha", "gamma", "lambda_R"] 
elseif model_tag_sym === :sliding
    global header_cont = ["beta", "alpha", "gamma"]            
else
    error("Unknown model tag: $(model_tag_sym)")
end

c = 1

@info "[$(String(model_tag_sym))_model] Fitting chain $(c) (tau=$tau)"

Random.seed!(2025 + c)
initθ_chain = initθ_for_chain(model_tag_sym) 
t0 = Dates.now()

try
    samples, loglik_aug_vecs =
        mcmc_one_chain_with_Rstar!(Istar_obs, N,S0, I0;
            fit_mech=model_tag_sym,
            n_iter=n_iter,
            initθ=initθ_chain,
            KMAX_UPPER=KMAX_UPPER,
            k_max_fixed = KMAX_fixed)
    if size(samples, 1) != n_iter
        error("Chain $c did not complete all iterations.")
    end

    # Save samples
    samples_filename = "samples_chain_$(c).csv"
    write_csv(joinpath(out_dir, samples_filename), header_cont, samples)

    # Save per-time log-likelihoods (thinned & post-burnin inside mcmc)
    loglik_filename = "loglik_chain_$(c).csv"
    write_csv(joinpath(out_dir, loglik_filename), ["loglik"], hcat(loglik_aug_vecs))
    el = Dates.value(Dates.now() - t0) / 1000
catch err
    el = Dates.value(Dates.now() - t0) / 1000
end

@info "Chain completed -> output dir: $out_dir"


[ Info: [sliding_model] Fitting chain 1 (tau=59)
[ Info: [sliding] iter 1000/1000000 elapsed=5.5s, rate=0.031, mean=[1.024, 0.00014, 0.356], std=[0.0084, 0.000295, 0.0036] [ADAPT]
[ Info: [sliding] iter 2000/1000000 elapsed=10.1s, rate=0.020, mean=[1.000, 0.00011, 0.358], std=[0.0281, 0.000220, 0.0039] [ADAPT]
[ Info: [sliding] iter 3000/1000000 elapsed=13.9s, rate=0.021, mean=[0.915, 0.00012, 0.368], std=[0.1279, 0.000188, 0.0154] [ADAPT]
[ Info: [sliding] iter 4000/1000000 elapsed=17.8s, rate=0.021, mean=[0.830, 0.00014, 0.378], std=[0.1716, 0.000171, 0.0207] [ADAPT]
[ Info: [sliding] iter 5000/1000000 elapsed=21.6s, rate=0.019, mean=[0.774, 0.00015, 0.384], std=[0.1832, 0.000160, 0.0214] [ADAPT]
[ Info: [sliding] iter 6000/1000000 elapsed=25.4s, rate=0.021, mean=[0.736, 0.00016, 0.388], std=[0.1839, 0.000152, 0.0210] [ADAPT]
[ Info: [sliding] iter 7000/1000000 elapsed=29.2s, rate=0.021, mean=[0.708, 0.00017, 0.391], std=[0.1810, 0.000146, 0.0209] [ADAPT]
[ Info: [sliding] iter 8000/